In [2]:
print("Hello world")

Hello world


In [ ]:
%pip install matplotlib pandas scipy

In [ ]:
%pip install opencv-python

In [ ]:
%pip install tqdm

In [ ]:
%pip install ipywidgets

In [ ]:
%pip install --upgrade pip

In [ ]:
%mkdir -p data/Image_data/

In [ ]:
%pip install torch torchvision --index-url https://download.pytorch.org/whl/cu126

In [ ]:
%pip install split-folders

In [3]:
import math
import numpy as np

def executable_to_rgb(file_path):
    """
    Converts an executable file into an RGB image representation 
    based on Algorithm 5.1.
    """
    C = 3 # Number of channels (RGB)
    
    # Read the executable file as raw bytes
    with open(file_path, 'rb') as f:
        file_bytes = f.read()
        
    # S <- FileSize(F)
    S = len(file_bytes)
    
    # R <- ceiling(sqrt(S)) // Image height/width
    R = math.ceil(math.sqrt(S))
    
    # N <- R * R
    N = R * R
    
    # Pad_Size <- N - S // Number of padding bytes
    Pad_Size = N - S
    
    # Create 1D array A and read file bytes into A. 
    # In Python, reading 'rb' directly gives us 8-bit values [0, 255].
    # We use np.frombuffer for efficient conversion to a 1D array.
    A = np.frombuffer(file_bytes, dtype=np.uint8)
    
    # Pad A with Pad_Size bytes of zeros to reach length N if needed
    if Pad_Size > 0:
        A = np.pad(A, (0, Pad_Size), mode='constant', constant_values=0)
        
    # Create 3D array RGB_Img of size R x R x C
    RGB_Img = np.zeros((R, R, C), dtype=np.uint8)
    
    # Copy the bytes in A into the 0-th channel of RGB_Img.
    # Note: The algorithm shows explicit nested for-loops (i and j). 
    # In Python, we use numpy's `.reshape()` to do this instantly without slow loops.
    RGB_Img[:, :, 0] = A.reshape((R, R))
    
    # The algorithm mentions converting 8-bit groups to decimal integers [0, 255].
    # Our numpy array using `dtype=np.uint8` handles this inherently!
    
    # Replicate the values across all C channels to generate RGB_Img
    RGB_Img[:, :, 1] = RGB_Img[:, :, 0]
    RGB_Img[:, :, 2] = RGB_Img[:, :, 0]
    
    return RGB_Img

In [ ]:
import cv2

In [ ]:
rgbimg = executable_to_rgb('./data/ELF_data/Hello_ELF')

cv2.imshow('RGB Image', rgbimg)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [ ]:
from pathlib import Path
from tqdm.notebook import tqdm  # Use tqdm.notebook for a cleaner bar in Jupyter

def convert_dataset_to_images(input_dir, output_dir):
    """
    Crawls the input directory, converts files, and saves them to the output directory
    while maintaining the Benignware/Malware folder structure.
    """
    input_path = Path(input_dir)
    output_path = Path(output_dir)
    
    # We expect these two folders based on your structure
    categories = ['Benignware', 'Malware']
    
    for category in categories:
        cat_input_dir = input_path / category
        cat_output_dir = output_path / category
        
        # Create the destination folder (e.g., data/Image_data/ELF_Dataset/Malware)
        cat_output_dir.mkdir(parents=True, exist_ok=True)
        
        # Grab all files in the current category
        if not cat_input_dir.exists():
            print(f"Warning: Directory {cat_input_dir} not found. Skipping.")
            continue
            
        files = [f for f in cat_input_dir.iterdir() if f.is_file()]
        print(f"Starting conversion for {category} ({len(files)} files)...")
        
        # Wrap the file list in tqdm for the progress bar
        for file_path in tqdm(files, desc=f"Processing {category}"):
            try:
                # 1. Generate the image array
                img_array = executable_to_rgb(file_path)
                
                # 2. Define the new file name (keep original name, change extension to .png)
                save_name = file_path.stem + ".png"
                save_path = cat_output_dir / save_name
                
                # 3. Save the image to the new directory
                cv2.imwrite(str(save_path), img_array)
                
            except Exception as e:
                # If an individual file fails (e.g., permission denied, 0 bytes), 
                # print the error and continue to the next file.
                print(f"Skipped {file_path.name} due to error: {e}")

# === Setup your paths ===
# Based on your root structure, we read from ELF_data and create a new Image_data folder
INPUT_DIRECTORY = "data/ELF_data/ELF_Dataset"
OUTPUT_DIRECTORY = "data/Image_data/ELF_Dataset" 

# Run the process
convert_dataset_to_images(INPUT_DIRECTORY, OUTPUT_DIRECTORY)
print("Dataset conversion complete!")